In [2]:
import torch
torch.set_printoptions(precision=4, sci_mode=False)

In [5]:
def convert_cellboxes(predictions, S=7, C=20, B=2, anchors=None):
    predictions = predictions.to("cpu")
    batch_size = predictions.shape[0]
    predictions = predictions.reshape(batch_size, S, S, C + 5 * B)

    if anchors is None:
        anchors = torch.tensor([[1.0, 1.0]] * B, dtype=predictions.dtype)
    anchors = anchors.to(predictions.dtype)

    pred_class_logits = predictions[..., :C]
    pred_anchor = predictions[..., C:].reshape(batch_size, S, S, B, 5)

    pred_obj = torch.sigmoid(pred_anchor[..., 0])
    pred_tx_ty = torch.sigmoid(pred_anchor[..., 1:3])
    pred_tw_th = pred_anchor[..., 3:5]

    anchor_wh = anchors.view(1, 1, 1, B, 2)
    pred_wh_cell = torch.exp(pred_tw_th) * anchor_wh

    cell_x = torch.arange(S).repeat(batch_size, S, 1).unsqueeze(-1).unsqueeze(-1)
    cell_y = cell_x.permute(0, 2, 1, 3, 4)

    pred_x = (pred_tx_ty[..., 0:1] + cell_x) / S
    pred_y = (pred_tx_ty[..., 1:2] + cell_y) / S
    pred_wh = pred_wh_cell / S
    pred_box_xywh = torch.cat((pred_x, pred_y, pred_wh), dim=-1)

    # BƯỚC 1 — Xác định lớp tốt nhất cho mỗi cell
    # Convert logits to probabilities using softmax
    pred_class_probs = torch.softmax(pred_class_logits, dim=-1)  # (batch, S, S, C)

    # Find best class_id and max probability for each cell
    class_id = torch.argmax(pred_class_probs, dim=-1, keepdim=True)  # (batch, S, S, 1)
    p_class_max = torch.max(pred_class_probs, dim=-1, keepdim=True)[0]  # (batch, S, S, 1)

    # Calculate score for each anchor
    scores = pred_obj * p_class_max  # (batch, S, S, B) - broadcasts (batch, S, S, B) * (batch, S, S, 1)

    # BƯỚC 2 — Expand class for all anchors
    class_id_expanded = class_id.unsqueeze(-2).expand(batch_size, S, S, B, 1)  # (batch, S, S, B, 1)

    # BƯỚC 3 — Prepare score tensor
    scores = scores.unsqueeze(-1)  # (batch, S, S, B, 1)

    # BƯỚC 4 — Concatenate final tensor [class_id, score, b_x, b_y, b_w, b_h]
    converted_preds = torch.cat([class_id_expanded.float(), scores, pred_box_xywh], dim=-1)  # (batch, S, S, B, 6)

    return converted_preds

In [8]:
# Test convert_cellboxes
BATCH_SIZE, S, C, B = 2, 7, 20, 2
anchors = torch.tensor([[1.0, 1.0], [2.0, 1.5]], dtype=torch.float32)
predictions = torch.linspace(
    -1.0, 1.0, steps=BATCH_SIZE * S * S * (C + 5 * B)
).reshape(BATCH_SIZE, S * S * (C + 5 * B))

converted = convert_cellboxes(predictions, S=S, C=C, B=B, anchors=anchors)
print("convert_cellboxes output shape:\n", converted.shape)
print("sample output[0,0,0]:\n", converted[0, 0, 0])

convert_cellboxes output shape:
 torch.Size([2, 7, 7, 2, 6])
sample output[0,0,0]:
 tensor([[19.0000,  0.0137,  0.0388,  0.0388,  0.0534,  0.0534],
        [19.0000,  0.0137,  0.0389,  0.0389,  0.1071,  0.0804]])


In [9]:
def cellboxes_to_boxes(out, S=7, C=20, B=2, anchors=None, threshold=0.5):
    converted_pred = convert_cellboxes(out, S=S, C=C, B=B, anchors=anchors).reshape(
        out.shape[0], S * S * B, 6
    )
    all_bboxes = []

    for ex_idx in range(out.shape[0]):
        # BƯỚC 1 — Lấy các box của ảnh hiện tại
        boxes = converted_pred[ex_idx]  # shape (S*S*B, 6)

        # BƯỚC 2 — Lọc các box có score > threshold
        boxes = boxes[boxes[:, 1] > threshold]  # shape (num_kept_boxes, 6)

        # BƯỚC 3 — Chuyển các box còn lại sang Python list
        boxes = boxes.tolist()

        # BƯỚC 4 — Ép class_id về kiểu int
        for box in boxes:
            box[0] = int(box[0])

        # BƯỚC 5 — Thêm danh sách box của ảnh hiện tại vào all_bboxes
        all_bboxes.append(boxes)

    return all_bboxes


boxes = cellboxes_to_boxes(predictions, S=S, C=C, B=B, anchors=anchors, threshold=0.01)
print("Number of images:", len(boxes))
print("Boxes for image 0:", len(boxes[0]), "boxes with score > 0.01")
print("Sample box:", boxes[0][0] if len(boxes[0]) > 0 else "No boxes")

Number of images: 2
Boxes for image 0: 98 boxes with score > 0.01
Sample box: [19, 0.013669264502823353, 0.03882291167974472, 0.038842152804136276, 0.0533832311630249, 0.0534195713698864]
